# Trino from a user's pod

Checks, from where a user runs Trino, that Trino takes the user's `OIDC_TOKEN`, with the pod's own Python client and the coordinator's own TLS certificate, and that the token can read a table:

1. Trino takes the user's token: a connection over HTTPS, as the user, runs a query;
2. the user's token can read a table of the catalog.

Each check is a query, so running the notebook fails on any error.

## Requirements

`users-acl.csv` (in `$AFON_HOME/config`) must authorise the test user for `AFON_CATALOG` (default `aws`), as for `stack-spark-connects`; after changing the ACL, apply it with `afon-cli.sh configure-users <cloud> all`.

`AFON_CATALOG` must also contain the table `AFON_TABLE` (default `admin.deployments`), written by the `spark-polaris` smoke test, which must have been run on the stack first.

It runs in a user's Jupyter pod, which has the Trino service `trino` at port 8443 and the certificate of its coordinator at `/etc/trino-tls/ca.crt`. It is not run from your machine, as Trino is not forwarded to it in non-project mode.

## Running

In a JupyterHub pod, `OIDC_TOKEN` is already injected: upload the notebook in JupyterLab and run all the cells. To run it in the pod of `USER_ID` of a deployed stack, use `afon-cli.sh smoke-notebook [<cloud>] stack-trino`, which gets a token for `USER_ID`, has the hub start their pod, runs the notebook in it and writes the executed notebook to `$AFON_HOME/smoke/notebooks`. It also runs in the smoke suite, as `notebooks`.

Settings, all optional apart from `OIDC_TOKEN`: `AFON_CATALOG` (which `smoke-notebook` sets to that of the cloud), `AFON_TABLE` (as `<namespace>.<table>`), `AFON_TRINO_HOST`, `AFON_TRINO_PORT` and `TRINO_CA_CERT`.

In [ ]:
import os

OIDC_TOKEN = os.environ["OIDC_TOKEN"]
CATALOG    = os.environ.get("AFON_CATALOG", "aws")
TABLE      = os.environ.get("AFON_TABLE", "admin.deployments")
TRINO_HOST = os.environ.get("AFON_TRINO_HOST", "trino")        # in the cluster, the service
TRINO_PORT = int(os.environ.get("AFON_TRINO_PORT", "8443"))    # not TRINO_PORT, which Kubernetes sets in a pod, to tcp://<ip>:<port>, for the service trino
TRINO_CA   = os.environ.get("TRINO_CA_CERT", "/etc/trino-tls/ca.crt")   # the coordinator's certificate, which is self-signed

## 1. Trino takes the user's token

Over HTTPS, trusting the coordinator's own certificate.

In [ ]:
from trino.auth import JWTAuthentication
from trino.dbapi import connect

NAMESPACE = TABLE.split(".")[0]

conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    catalog=CATALOG,
    schema=NAMESPACE,
    http_scheme="https",
    auth=JWTAuthentication(OIDC_TOKEN),
    extra_credential=[("token", OIDC_TOKEN)],
    verify=TRINO_CA,
)

cursor = conn.cursor()
cursor.execute("SELECT current_catalog")
print(cursor.fetchone()) # this should work

## 2. The user's token can read a table

In [ ]:
cursor.execute(f"SELECT * FROM {CATALOG}.{TABLE} LIMIT 5")
print(cursor.fetchall())